# 🌿 Détection des Maladies des Plantes — Deep Learning
## ÉTAPE 5/9 : Fine-Tuning ResNet50 (Transfer Learning)

**Dataset :** PlantVillage — 54 305 images RGB (224×224×3), 38 classes  
**Objectif :** Entraîner ResNet50 avec Fine-Tuning et réaliser un benchmarking complet à trois voies : CNN From Scratch / MobileNetV2 / ResNet50.

---

### 📋 Plan du notebook

| Partie | Contenu |
|--------|---------|
| 0 | Reconstruction des générateurs avec normalisation ResNet50 |
| 1 | Chargement de ResNet50 pré-entraîné (ImageNet) |
| 2 | Construction du modèle complet + bilan des paramètres |
| 3 | Phase 1 — Feature Extraction (base gelée, 5 epochs) |
| 4 | Phase 2 — Fine-Tuning (30 dernières couches, LR=1e-5, 10 epochs) |
| 5 | Évaluation complète (Accuracy, Loss, Precision, Recall, F1) |
| 6 | Visualisations (courbes, matrice de confusion, classification report) |
| 7 | Benchmarking 3 modèles : CNN From Scratch vs MobileNetV2 vs ResNet50 |
| 8 | Sauvegarde (`.keras` et `.h5`) |
| 9 | Analyse professionnelle complète |

> 💡 Chaque partie est précédée d'une explication scientifique détaillée. Les lignes de code importantes sont commentées directement dans les cellules.  
> ⚠️ Ce notebook fonctionne de manière **autonome** : il recharge les artefacts de l'étape 2 et les métriques des étapes 3 & 4.


## 0. ⚙️ Configuration et reconstruction des générateurs

### Normalisation spécifique à ResNet50

Chaque architecture de réseau pré-entraîné exige **sa propre normalisation** — appliquer la mauvaise normalisation peut coûter plusieurs points d'accuracy car les statistiques d'activation de la base ne correspondent plus à celles vues pendant l'entraînement ImageNet.

| Modèle | Fonction de normalisation | Plage de sortie |
|--------|--------------------------|-----------------|
| CNN From Scratch | `rescale=1/255` | [0, 1] |
| MobileNetV2 | `mobilenet_v2.preprocess_input` | [-1, 1] |
| **ResNet50** | **`resnet50.preprocess_input`** | **[−103.9, 151.1] (BGR centré sur ImageNet)** |

`resnet50.preprocess_input` applique deux opérations :  
1. Conversion RGB → BGR (inversion des canaux)  
2. Soustraction des moyennes ImageNet pixel à pixel : [103.939, 116.779, 123.680]


In [ ]:
# ─── Installations silencieuses ───────────────────────────────────────────────
!pip install -q tensorflow scikit-learn seaborn pillow

import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ─── Vérification GPU ─────────────────────────────────────────────────────────
gpu_devices = tf.config.list_physical_devices("GPU")
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU détecté(s)     : {gpu_devices if gpu_devices else '⚠️ AUCUN GPU — activez-le dans Runtime > Change runtime type'}")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
print("✅ Seed fixée à 42 pour la reproductibilité")


In [ ]:
# ─── Chargement des artefacts de l'étape 2 ────────────────────────────────────
SAVE_DIR = "preprocessing_artifacts"

with open(os.path.join(SAVE_DIR, "class_names.json"), "r", encoding="utf-8") as f:
    class_info = json.load(f)

with open(os.path.join(SAVE_DIR, "preprocessing_config.json"), "r", encoding="utf-8") as f:
    preprocessing_config = json.load(f)

class_names = class_info["class_names"]
num_classes  = class_info["num_classes"]

train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_df.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_df.csv"))
test_df  = pd.read_csv(os.path.join(SAVE_DIR, "test_df.csv"))

IMG_SIZE   = tuple(preprocessing_config["img_size"])   # (224, 224)
BATCH_SIZE = preprocessing_config["batch_size"]        # 32

print(f"✅ Artefacts chargés : {num_classes} classes")
print(f"   Train      : {len(train_df):,} images")
print(f"   Validation : {len(val_df):,} images")
print(f"   Test       : {len(test_df):,} images")
print(f"   IMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE}")


In [ ]:
# ─── Reconstruction des générateurs avec resnet_preprocess ────────────────────
# ResNet50 attend une normalisation BGR centrée sur ImageNet (pas [0,1] ni [-1,1])

resnet_train_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,  # normalisation spécifique ResNet50
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

resnet_eval_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,  # même normalisation, sans augmentation
)

common_args = dict(
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=SEED,
)

resnet_train_gen = resnet_train_datagen.flow_from_dataframe(train_df, shuffle=True,  **common_args)
resnet_val_gen   = resnet_eval_datagen.flow_from_dataframe(val_df,   shuffle=False, **common_args)
resnet_test_gen  = resnet_eval_datagen.flow_from_dataframe(test_df,  shuffle=False, **common_args)

# Vérification cohérence encodage classes
assert resnet_train_gen.class_indices == class_info["class_indices"], \
    "⚠️ Incohérence d'encodage des classes — vérifiez le SAVE_DIR et l'étape 2 !"

print("✅ Générateurs ResNet50 reconstruits avec succès.")
print("ℹ️  Normalisation active : resnet50.preprocess_input → BGR centré ImageNet")


## 🏗️ PARTIE 1 — Chargement du modèle de base ResNet50

### Qu'est-ce que ResNet50 ?

ResNet50 (He et al., 2016) est l'un des réseaux de neurones convolutifs les plus influents de l'histoire du Deep Learning. Son innovation majeure est l'introduction des **connexions résiduelles** (skip connections) qui permettent de construire des réseaux très profonds sans souffrir du problème de la **disparition du gradient** (vanishing gradient).

| Caractéristique | MobileNetV2 | ResNet50 |
|---|---|---|
| **Profondeur** | 53 couches | 50 couches (+ skip connections) |
| **Paramètres** | ~3,4 M | ~25,6 M |
| **Blocs clés** | Inverted Residual + Depthwise Sep. Conv | Bottleneck Residual Blocks |
| **Top-1 Accuracy ImageNet** | ~71,8% | ~74,9% |
| **Mémoire GPU** | Légère | Moyenne |
| **Vitesse d'inférence** | Rapide (mobile) | Modérée |

### Principe des connexions résiduelles (skip connections)

```
Input ──┬────────────────────────────► (+) ──► Output
        │  Conv → BN → ReLU             ▲
        │  Conv → BN → ReLU             │
        └────────────────────────────────┘
              (shortcut / skip)
```

L'idée : plutôt qu'apprendre une transformation `F(x)`, le bloc apprend le **résidu** `F(x) = H(x) − x`. Si la transformation optimale est proche de l'identité, il suffit d'apprendre `F(x) ≈ 0`, ce qui est plus facile. Cela permet à l'information de traverser les couches profondes sans dégradation du gradient.

> **Pourquoi 30 couches dégelées (vs 20 pour MobileNetV2) ?**  
> ResNet50 a des blocs Bottleneck de 3 couches chacun. Dégeler 30 couches permet d'affiner ~10 blocs complets tout en préservant les features génériques des couches initiales. Avec MobileNetV2 (blocs plus fins), 20 couches suffisaient.


In [ ]:
# ─── Chargement de ResNet50 pré-entraîné sur ImageNet ─────────────────────────
base_model = ResNet50(
    weights="imagenet",        # poids pré-entraînés sur ImageNet (1,28 M d'images, 1 000 classes)
    include_top=False,          # on retire la tête Dense(1000) originale
    input_shape=(*IMG_SIZE, 3), # (224, 224, 3)
)

print(f"✅ ResNet50 chargé avec succès.")
print(f"   Nombre de couches dans la base : {len(base_model.layers)}")
print(f"   Shape de sortie de la base     : {base_model.output_shape}")
# ResNet50 produit une carte de features (None, 7, 7, 2048)
# → GlobalAveragePooling2D réduira à (None, 2048)
# → 2048 > 1280 (MobileNetV2) : features plus riches mais plus coûteuses

# Affichage des 5 premières et 5 dernières couches pour inspection
print("\n   5 premières couches :")
for l in base_model.layers[:5]:
    print(f"     [{l.name:<35}] output: {str(l.output_shape)}")
print("   ...")
print("   5 dernières couches :")
for l in base_model.layers[-5:]:
    print(f"     [{l.name:<35}] output: {str(l.output_shape)}")


## 🔧 PARTIE 2 — Construction du modèle complet

### Architecture de la tête de classification

| Couche | Paramètres | Rôle |
|--------|-----------|------|
| **GlobalAveragePooling2D** | 0 | Réduit (7×7×2048) → (2048) par moyennage spatial — compact et efficace |
| **Dense(512, relu)** | 2 048×512 + 512 = **1 049 088** | Couche de décision intermédiaire |
| **BatchNormalization** | 512×4 = 2 048 | Stabilise l'entraînement, normalise les activations |
| **Dropout(0.5)** | 0 | Régularisation : 50% des neurones désactivés aléatoirement |
| **Dense(38, softmax)** | 512×38 + 38 = **19 494** | Sortie : probabilité sur 38 classes |

> **Comparaison des têtes :**  
> - MobileNetV2 : base produit 1 280 features → Dense(512) ≈ 655 K paramètres dans la tête  
> - ResNet50 : base produit **2 048** features → Dense(512) ≈ **1 049 K** paramètres dans la tête  
> La tête ResNet50 est donc plus riche, ce qui lui confère potentiellement plus de capacité discriminante pour 38 classes fines.


In [ ]:
# ─── Construction du modèle complet : ResNet50 + tête de classification ─────────
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name="input_224")

# Passage dans la base ResNet50 (feature extraction)
# training=False → les couches BatchNorm de la base restent en mode inférence (important !)
x = base_model(inputs, training=False)

# ── Tête de classification personnalisée ──────────────────────────────────────
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
# (None, 7, 7, 2048) → (None, 2048) par moyennage spatial

x = layers.Dense(512, activation="relu", name="dense_512")(x)
# Couche intermédiaire de décision

x = layers.BatchNormalization(name="bn_top")(x)
# Normalisation des activations de la tête pour une convergence stable

x = layers.Dropout(0.5, name="dropout_50")(x)
# Régularisation — réduit l'overfitting sur PlantVillage

outputs = layers.Dense(num_classes, activation="softmax", name="output_38")(x)
# 38 sorties avec softmax (probabilités)

resnet_model = tf.keras.Model(inputs, outputs, name="ResNet50_PlantDisease")

print("\n" + "=" * 70)
print(" RÉSUMÉ DU MODÈLE RESNET50 COMPLET")
print("=" * 70)
resnet_model.summary()


In [ ]:
# ─── Bilan complet des paramètres ─────────────────────────────────────────────
total_params      = resnet_model.count_params()
trainable_params  = int(np.sum([np.prod(v.shape) for v in resnet_model.trainable_variables]))
non_trainable     = total_params - trainable_params

print("\n" + "=" * 60)
print(" BILAN DES PARAMÈTRES — ResNet50")
print("=" * 60)
print(f"  Total                       : {total_params:>12,}")
print(f"  Entraînables (tête)         : {trainable_params:>12,}")
print(f"  Non entraînables (base)     : {non_trainable:>12,}")
print("=" * 60)

# Comparaison rapide avec MobileNetV2
print("\n  ℹ️  Rappel comparatif :")
print(f"  MobileNetV2 : ~3 400 000 paramètres totaux")
print(f"  ResNet50    : {total_params:,} paramètres totaux")
print(f"  ResNet50 est ~{total_params/3_400_000:.1f}x plus lourd que MobileNetV2")
print("=" * 60)


## 🧊 PARTIE 3 — Phase 1 : Feature Extraction (base gelée)

### Stratégie en deux phases — rappel

| Phase | Couches entraînées | LR | Epochs | Objectif |
|---|---|---|---|---|
| **1 — Feature Extraction** | Tête seulement (~1,07 M params) | `0.001` | 5 | Initialiser la tête sans toucher aux poids ImageNet |
| **2 — Fine-Tuning** | Tête + 30 dernières couches de base | `0.00001` | 10 max | Spécialiser les features profondes au domaine botanique |

**Pourquoi geler d'abord ?**  
Les gradients issus d'une tête non entraînée sont très bruités. Les propager dans la base risque de détruire les poids ImageNet soigneusement optimisés (**catastrophic forgetting**). La Phase 1 laisse la tête converger, puis la Phase 2 affine la base avec des gradients propres et un LR très bas.


In [ ]:
# ─── PHASE 1 : Gel de toutes les couches de la base ResNet50 ─────────────────
base_model.trainable = False   # aucune couche de la base ne sera mise à jour

# Vérification
n_trainable = int(np.sum([np.prod(v.shape) for v in resnet_model.trainable_variables]))
n_frozen    = resnet_model.count_params() - n_trainable

print("📊 Phase 1 — Bilan après gel :")
print(f"   Couches totales           : {len(resnet_model.layers)}")
print(f"   Paramètres entraînables   : {n_trainable:,}  (tête seulement)")
print(f"   Paramètres gelés          : {n_frozen:,}  (base ResNet50)")


In [ ]:
# ─── Calcul des poids de classes ──────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(resnet_train_gen.classes),
    y=resnet_train_gen.classes,
)
class_weight_dict = dict(enumerate(class_weights_array))

print("⚖️  Poids de classes calculés (5 premiers) :")
for i in list(class_weight_dict.keys())[:5]:
    print(f"   Classe {i:02d} ({class_names[i]:<35}) -> poids = {class_weight_dict[i]:.4f}")


In [ ]:
# ─── Compilation — Phase 1 ────────────────────────────────────────────────────
resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),  # LR standard pour la tête
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print("✅ Modèle compilé (Phase 1) : Adam(lr=0.001), categorical_crossentropy, accuracy")


In [ ]:
# ─── Entraînement — Phase 1 ────────────────────────────────────────────────────
EPOCHS_PHASE1    = 5
steps_per_epoch  = resnet_train_gen.samples  // BATCH_SIZE
validation_steps = resnet_val_gen.samples    // BATCH_SIZE

callbacks_phase1 = [
    ModelCheckpoint(
        filepath="best_resnet50_phase1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

print("🚀 Début Phase 1 — Feature Extraction (ResNet50)")
print(f"   epochs={EPOCHS_PHASE1} | steps/epoch={steps_per_epoch} | val_steps={validation_steps}")
print("─" * 60)

start_phase1 = time.time()

history_phase1 = resnet_model.fit(
    resnet_train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=resnet_val_gen,
    validation_steps=validation_steps,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
    verbose=1,
)

time_phase1 = time.time() - start_phase1
print(f"\n✅ Phase 1 terminée en {time_phase1/60:.1f} min.")
print(f"   Meilleure val_accuracy (Phase 1) : {max(history_phase1.history['val_accuracy']):.4f}")


## 🔥 PARTIE 4 — Phase 2 : Fine-Tuning (30 dernières couches)

### Pourquoi 30 couches pour ResNet50 (vs 20 pour MobileNetV2) ?

ResNet50 est organisé en **4 stages** de blocs Bottleneck :

| Stage | Couches | Features encodées |
|-------|---------|-------------------|
| Stage 1 (conv1) | 1 conv | Contours bas niveau, gradients |
| Stage 2 (conv2_x) | 9 couches | Textures, coins, fréquences spatiales |
| Stage 3 (conv3_x) | 12 couches | Motifs intermédiaires (veines, taches) |
| Stage 4 (conv4_x) | 18 couches | Formes complexes, structures |
| **Stage 5 (conv5_x)** | **9 couches** | **Représentations sémantiques haut niveau** |

Dégeler les **30 dernières couches** capture le Stage 5 entier + la fin du Stage 4 — les niveaux les plus susceptibles de bénéficier d'une spécialisation botanique, tout en préservant les features génériques des Stages 1–3.

**`learning_rate = 0.00001` (1e-5)** : même ordre de grandeur que pour MobileNetV2 — suffisamment bas pour ne pas détruire les poids ImageNet du Stage 5, suffisamment élevé pour converger.


In [ ]:
# ─── PHASE 2 : Dégel des 30 dernières couches de ResNet50 ────────────────────
base_model.trainable = True   # autorise la modification des poids de la base

# On regèle les couches sauf les 30 dernières
for layer in base_model.layers[:-30]:
    layer.trainable = False   # couches bas niveau (génériques) → gelées

# Vérification
n_trainable_p2 = int(np.sum([np.prod(v.shape) for v in resnet_model.trainable_variables]))
n_frozen_p2    = resnet_model.count_params() - n_trainable_p2

print("📊 Phase 2 — Bilan après dégel des 30 dernières couches :")
print(f"   Paramètres entraînables : {n_trainable_p2:>12,}  (tête + 30 couches base)")
print(f"   Paramètres gelés        : {n_frozen_p2:>12,}")
print(f"   Total                   : {resnet_model.count_params():>12,}")

# Identification des couches dégelées dans la base
print("\n   Couches dégelées dans la base ResNet50 :")
unfrozen = [l.name for l in base_model.layers[-30:]]
print(f"   {unfrozen[:5]} ... {unfrozen[-3:]} (+ couches de la tête)")


In [ ]:
# ─── Compilation — Phase 2 ────────────────────────────────────────────────────
LR_FINETUNE = 1e-5  # LR très bas pour affinage délicat

resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"✅ Modèle recompilé (Phase 2) : Adam(lr={LR_FINETUNE})")


In [ ]:
# ─── Callbacks — Phase 2 ─────────────────────────────────────────────────────
EPOCHS_PHASE2 = 10

callbacks_phase2 = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,                  # tolère 5 epochs sans amélioration
        restore_best_weights=True,   # restaure automatiquement les meilleurs poids
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,                  # divise le LR par 5 en cas de stagnation
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath="best_resnet50_finetune.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]
print("✅ Callbacks Phase 2 : EarlyStopping(patience=5), ReduceLROnPlateau(factor=0.2), ModelCheckpoint")


In [ ]:
# ─── Entraînement — Phase 2 ────────────────────────────────────────────────────
initial_epoch = EPOCHS_PHASE1   # reprend après la Phase 1

print("🔥 Début Phase 2 — Fine-Tuning ResNet50")
print(f"   epochs={EPOCHS_PHASE2} | LR={LR_FINETUNE} | 30 dernières couches dégelées")
print("─" * 60)

start_phase2 = time.time()

history_phase2 = resnet_model.fit(
    resnet_train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=resnet_val_gen,
    validation_steps=validation_steps,
    epochs=initial_epoch + EPOCHS_PHASE2,
    initial_epoch=initial_epoch,           # continuation depuis la Phase 1
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
    verbose=1,
)

time_phase2 = time.time() - start_phase2
training_time_total = time_phase1 + time_phase2

print(f"\n✅ Phase 2 terminée en {time_phase2/60:.1f} min.")
print(f"✅ Temps total d'entraînement : {training_time_total/60:.1f} min.")
print(f"   Meilleure val_accuracy (Phase 2) : {max(history_phase2.history['val_accuracy']):.4f}")


In [ ]:
# ─── Fusion des historiques Phase 1 + Phase 2 ─────────────────────────────────
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history_full = merge_histories(history_phase1, history_phase2)
total_epochs_run = len(history_full["loss"])

print(f"✅ Historiques fusionnés : {total_epochs_run} epochs au total.")


## 📊 PARTIE 5 — Évaluation sur le jeu de test

Toutes les métriques sont calculées sur le **jeu de test** (données non vues pendant l'entraînement).

| Métrique | Aggregation | Interprétation |
|---|---|---|
| **Accuracy** | — | % global de prédictions correctes |
| **Precision** | macro / weighted | Taux de vrais positifs parmi les prédictions positives |
| **Recall** | macro / weighted | Taux de vrais positifs détectés parmi les cas réels |
| **F1-Score** | macro / weighted | Compromis Precision/Recall — robuste aux déséquilibres |

> **macro** : moyenne non pondérée (chaque classe pèse autant)  
> **weighted** : moyenne pondérée par la taille de chaque classe (représentativité globale)


In [ ]:
# ─── Prédictions sur le jeu de test ──────────────────────────────────────────
print("🔄 Calcul des prédictions sur le jeu de test...")

resnet_test_gen.reset()
y_pred_proba = resnet_model.predict(
    resnet_test_gen,
    steps=int(np.ceil(resnet_test_gen.samples / BATCH_SIZE)),
    verbose=1,
)

y_pred = np.argmax(y_pred_proba, axis=1)
y_true = resnet_test_gen.classes
y_pred = y_pred[:len(y_true)]   # ajustement si arrondi batch

print(f"\n✅ Prédictions calculées sur {len(y_true):,} images de test.")


In [ ]:
# ─── Calcul de toutes les métriques ───────────────────────────────────────────
test_loss, test_accuracy = resnet_model.evaluate(
    resnet_test_gen,
    steps=int(np.ceil(resnet_test_gen.samples / BATCH_SIZE)),
    verbose=0,
)

precision_macro    = precision_score(y_true, y_pred, average="macro",    zero_division=0)
recall_macro       = recall_score   (y_true, y_pred, average="macro",    zero_division=0)
f1_macro           = f1_score       (y_true, y_pred, average="macro",    zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted    = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
f1_weighted        = f1_score       (y_true, y_pred, average="weighted", zero_division=0)

print("\n" + "=" * 62)
print(" MÉTRIQUES D'ÉVALUATION — ResNet50 Fine-Tuned")
print("=" * 62)
print(f"  Loss (test)                  : {test_loss:.4f}")
print(f"  Accuracy (test)              : {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)")
print("─" * 62)
print(f"  Precision  (macro avg)       : {precision_macro:.4f}")
print(f"  Recall     (macro avg)       : {recall_macro:.4f}")
print(f"  F1-Score   (macro avg)       : {f1_macro:.4f}")
print("─" * 62)
print(f"  Precision  (weighted avg)    : {precision_weighted:.4f}")
print(f"  Recall     (weighted avg)    : {recall_weighted:.4f}")
print(f"  F1-Score   (weighted avg)    : {f1_weighted:.4f}")
print("=" * 62)


## 📈 PARTIE 6 — Visualisations

Quatre visualisations produites :
1. **Courbes Accuracy & Loss** (Phase 1 + Phase 2 fusionnées, ligne de séparation)
2. **Matrice de confusion** (38×38, normalisée en %)
3. **Classification Report** (par classe, graphique F1-score)
4. **Radar chart** Precision / Recall / F1 par classe (top/bottom 10)


In [ ]:
# ─── Courbes Accuracy et Loss ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
epochs_range = range(1, total_epochs_run + 1)
phase1_end   = EPOCHS_PHASE1

# ── Accuracy ──────────────────────────────────────────────────────────────────
axes[0].plot(epochs_range, history_full["accuracy"],     linewidth=2, color="#1565C0", label="Train Accuracy")
axes[0].plot(epochs_range, history_full["val_accuracy"], linewidth=2, color="#2E7D32", label="Validation Accuracy")
axes[0].axvline(x=phase1_end, color="#FF6F00", linestyle="--", linewidth=2,
                label=f"Fine-Tuning (epoch {phase1_end+1})")
axes[0].fill_betweenx([0, 1], 1, phase1_end,       alpha=0.06, color="#1565C0")
axes[0].fill_betweenx([0, 1], phase1_end, total_epochs_run, alpha=0.06, color="#B71C1C")
axes[0].text(phase1_end/2, 0.05, "Feature\nExtraction", ha="center", fontsize=9, color="#1565C0")
axes[0].text(phase1_end + (total_epochs_run-phase1_end)/2, 0.05, "Fine-Tuning",
             ha="center", fontsize=9, color="#B71C1C")
axes[0].set_title("Accuracy — ResNet50", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim([0, 1.05])
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── Loss ──────────────────────────────────────────────────────────────────────
axes[1].plot(epochs_range, history_full["loss"],     linewidth=2, color="#1565C0", label="Train Loss")
axes[1].plot(epochs_range, history_full["val_loss"], linewidth=2, color="#C62828", label="Validation Loss")
axes[1].axvline(x=phase1_end, color="#FF6F00", linestyle="--", linewidth=2,
                label=f"Fine-Tuning (epoch {phase1_end+1})")
axes[1].set_title("Loss — ResNet50", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("resnet50_courbes_entrainement.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 resnet50_courbes_entrainement.png")


In [ ]:
# ─── Matrice de confusion (38x38, normalisée) ─────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(22, 20))
sns.heatmap(
    cm_normalized,
    annot=False,
    cmap="YlOrRd",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "% de prédictions par classe réelle"},
    linewidths=0.1,
    linecolor="lightgray",
)
plt.title("Matrice de confusion (normalisée, %) — ResNet50 Fine-Tuned",
          fontsize=15, fontweight="bold")
plt.xlabel("Classe prédite", fontsize=12)
plt.ylabel("Classe réelle", fontsize=12)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0,  fontsize=6)
plt.tight_layout()
plt.savefig("resnet50_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Top 5 confusions hors diagonale
cm_no_diag = cm_normalized.copy()
np.fill_diagonal(cm_no_diag, 0)
top5_idx = np.dstack(np.unravel_index(np.argsort(-cm_no_diag.ravel()), cm_no_diag.shape))[0][:5]

print("💾 resnet50_confusion_matrix.png")
print("\n🔍 Top 5 confusions inter-classes (ResNet50) :")
for ti, pi in top5_idx:
    if cm_no_diag[ti, pi] > 0:
        print(f"   {class_names[ti]:<40} → {class_names[pi]:<40} ({cm_no_diag[ti,pi]:.1f}%)")


In [ ]:
# ─── Classification Report + graphique classes difficiles ─────────────────────
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()

print("📋 Classification Report (par classe) :\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

report_df.to_csv("resnet50_classification_report.csv")
print("💾 resnet50_classification_report.csv")

# ── Graphique double : 10 meilleures et 10 moins bonnes classes ───────────────
per_class = report_df.iloc[:-3].copy()   # exclut accuracy/macro/weighted

worst10 = per_class.sort_values("f1-score").head(10)
best10  = per_class.sort_values("f1-score").tail(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Moins bonnes
bars_w = axes[0].barh(worst10.index, worst10["f1-score"],
                      color=plt.cm.Reds(np.linspace(0.4, 0.8, 10)))
axes[0].set_xlabel("F1-Score")
axes[0].set_title("10 classes les plus difficiles", fontsize=12, fontweight="bold")
for bar, val in zip(bars_w, worst10["f1-score"]):
    axes[0].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=8)

# Meilleures
bars_b = axes[1].barh(best10.index, best10["f1-score"],
                      color=plt.cm.Greens(np.linspace(0.4, 0.8, 10)))
axes[1].set_xlabel("F1-Score")
axes[1].set_title("10 classes les mieux reconnues", fontsize=12, fontweight="bold")
for bar, val in zip(bars_b, best10["f1-score"]):
    axes[1].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=8)

plt.suptitle("F1-Score par classe — ResNet50 Fine-Tuned", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("resnet50_classes_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 resnet50_classes_f1.png")


## ⚖️ PARTIE 7 — Benchmarking complet : CNN From Scratch vs MobileNetV2 vs ResNet50

### Critères de comparaison

| Critère | Interprétation |
|---|---|
| **Accuracy** | Performance globale sur le jeu de test |
| **Loss** | Plus bas = meilleures probabilités calibrées |
| **F1-Score (weighted)** | Performance tenant compte du déséquilibre des classes |
| **Nombre de paramètres** | Complexité du modèle, empreinte mémoire |
| **Temps d'entraînement** | Coût computationnel GPU |
| **Taille du fichier** | Contrainte de déploiement |


In [ ]:
# ─── Chargement des métriques des étapes précédentes ──────────────────────────
def load_metrics(path, default_name):
    if os.path.exists(path):
        with open(path) as f:
            m = json.load(f)
        print(f"✅ {path} chargé.")
        return m, True
    else:
        print(f"⚠️  {path} introuvable — valeurs à 0. Ré-exécutez l'étape correspondante.")
        return {"model_name": default_name, "test_accuracy": 0, "test_loss": 0,
                "f1_weighted": 0, "total_params": 0,
                "training_time_minutes": 0, "epochs_executed": 0}, False

cnn_metrics,     cnn_ok     = load_metrics("cnn_final_metrics.json",     "CNN_From_Scratch")
mobilenet_metrics, mob_ok   = load_metrics("mobilenet_final_metrics.json", "MobileNetV2_FineTuned")

# Métriques ResNet50 (calculées dans ce notebook)
resnet_metrics = {
    "model_name"             : "ResNet50_FineTuned",
    "test_accuracy"          : float(test_accuracy),
    "test_loss"              : float(test_loss),
    "precision_macro"        : float(precision_macro),
    "recall_macro"           : float(recall_macro),
    "f1_macro"               : float(f1_macro),
    "precision_weighted"     : float(precision_weighted),
    "recall_weighted"        : float(recall_weighted),
    "f1_weighted"            : float(f1_weighted),
    "total_params"           : int(resnet_model.count_params()),
    "training_time_minutes"  : round(training_time_total / 60, 2),
    "epochs_executed"        : total_epochs_run,
}
print(f"✅ Métriques ResNet50 collectées.")


In [ ]:
# ─── Taille des fichiers sauvegardés ──────────────────────────────────────────
def get_size_mb(path):
    return round(os.path.getsize(path) / 1e6, 1) if os.path.exists(path) else "N/A"

# Sauvegarde temporaire pour mesurer la taille (sera refaite proprement en Partie 8)
resnet_model.save("resnet50_plant_disease.keras")
resnet_model.save("resnet50_plant_disease.h5")

sizes = {
    "CNN From Scratch"       : get_size_mb("cnn_plant_disease.keras"),
    "MobileNetV2 Fine-Tuned" : get_size_mb("mobilenetv2_plant_disease.keras"),
    "ResNet50 Fine-Tuned"    : get_size_mb("resnet50_plant_disease.keras"),
}
print("📦 Tailles des modèles (.keras) :")
for name, size in sizes.items():
    print(f"   {name:<30} : {size} MB")


In [ ]:
# ─── Tableau de benchmarking complet ──────────────────────────────────────────
models_list = [
    ("CNN From Scratch",       cnn_metrics),
    ("MobileNetV2 Fine-Tuned", mobilenet_metrics),
    ("ResNet50 Fine-Tuned",    resnet_metrics),
]

rows = []
for name, m in models_list:
    rows.append({
        "Modèle"                      : name,
        "Accuracy"                    : f"{m.get('test_accuracy', 0):.4f}",
        "Loss"                        : f"{m.get('test_loss', 0):.4f}",
        "F1-Score (weighted)"         : f"{m.get('f1_weighted', 0):.4f}",
        "Nb Paramètres"               : f"{m.get('total_params', 0):,}",
        "Temps entraînement (min)"    : f"{m.get('training_time_minutes', 0):.1f}",
        "Taille modèle (.keras)"      : f"{sizes.get(name, 'N/A')} MB",
    })

bench_df = pd.DataFrame(rows).set_index("Modèle")

print("\n" + "=" * 90)
print(" TABLEAU DE BENCHMARKING — 3 MODÈLES")
print("=" * 90)
print(bench_df.to_string())
print("=" * 90)

bench_df.to_csv("benchmarking_3_modeles.csv")
print("\n💾 benchmarking_3_modeles.csv")


In [ ]:
# ─── Graphiques de benchmarking (4 métriques) ────────────────────────────────
model_labels = ["CNN\nFrom Scratch", "MobileNetV2\nFine-Tuned", "ResNet50\nFine-Tuned"]
colors       = ["#90CAF9", "#A5D6A7", "#FFCC80"]

accs    = [m.get("test_accuracy",0)          for _, m in models_list]
losses  = [m.get("test_loss",0)              for _, m in models_list]
f1s     = [m.get("f1_weighted",0)            for _, m in models_list]
times   = [m.get("training_time_minutes",0)  for _, m in models_list]
params  = [m.get("total_params",0) / 1e6     for _, m in models_list]  # en millions

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

def bar_chart(ax, values, title, ylabel, fmt="{:.4f}", highlight_max=True):
    bars = ax.bar(model_labels, values, color=colors, edgecolor="gray", linewidth=0.7)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.3)
    best = max(values) if highlight_max else min(values)
    for bar, val in zip(bars, values):
        color = "#1B5E20" if val == best else "#333"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                fmt.format(val), ha="center", fontsize=10, fontweight="bold", color=color)

bar_chart(axes[0], accs,   "Accuracy",              "Accuracy")
bar_chart(axes[1], losses, "Loss",                  "Loss",     fmt="{:.4f}", highlight_max=False)
bar_chart(axes[2], f1s,    "F1-Score (weighted)",   "F1-Score")
bar_chart(axes[3], params, "Paramètres (millions)", "Millions de params", fmt="{:.1f}M", highlight_max=False)

plt.suptitle("Benchmarking : CNN From Scratch vs MobileNetV2 vs ResNet50",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("benchmarking_3_modeles.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 benchmarking_3_modeles.png")


In [ ]:
# ─── Graphique Temps vs Accuracy (scatter plot) ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

scatter_colors = ["#1565C0", "#2E7D32", "#E65100"]
for i, (name, m) in enumerate(models_list):
    ax.scatter(
        times[i], accs[i],
        s=params[i] * 30,   # taille du cercle proportionnelle aux paramètres
        color=scatter_colors[i],
        alpha=0.85,
        edgecolors="white",
        linewidths=1.5,
        label=f"{model_labels[i].replace(chr(10), ' ')} ({params[i]:.1f}M params)",
        zorder=3,
    )
    ax.annotate(
        model_labels[i].replace("\n", "\n"),
        (times[i], accs[i]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=9,
        color=scatter_colors[i],
        fontweight="bold",
    )

ax.set_xlabel("Temps d'entraînement (min)", fontsize=12)
ax.set_ylabel("Accuracy (test)", fontsize=12)
ax.set_title("Temps vs Accuracy (taille du cercle ∝ Nb paramètres)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("benchmarking_temps_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 benchmarking_temps_vs_accuracy.png")


## 💾 PARTIE 8 — Sauvegarde du modèle ResNet50

**Deux formats sauvegardés :**

| Format | Extension | Usage recommandé |
|---|---|---|
| **Keras natif** | `.keras` | Déploiement Keras/TF >= 2.12, format optimal |
| **HDF5** | `.h5` | Compatibilité universelle, anciens outils, Streamlit, Hugging Face |


In [ ]:
# ─── Sauvegarde finale ────────────────────────────────────────────────────────
resnet_model.save("resnet50_plant_disease.keras")
print("💾 resnet50_plant_disease.keras")

resnet_model.save("resnet50_plant_disease.h5")
print("💾 resnet50_plant_disease.h5")

# Historique et métriques
with open("resnet50_training_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history_full.items()}, f, indent=2)
print("💾 resnet50_training_history.json")

with open("resnet50_final_metrics.json", "w") as f:
    json.dump(resnet_metrics, f, indent=2)
print("💾 resnet50_final_metrics.json")

# Récapitulatif de tous les fichiers générés
print("\n" + "=" * 60)
print(" FICHIERS GÉNÉRÉS — ÉTAPE 5")
print("=" * 60)
files = [
    "resnet50_plant_disease.keras",
    "resnet50_plant_disease.h5",
    "best_resnet50_phase1.keras",
    "best_resnet50_finetune.keras",
    "resnet50_training_history.json",
    "resnet50_final_metrics.json",
    "resnet50_courbes_entrainement.png",
    "resnet50_confusion_matrix.png",
    "resnet50_classes_f1.png",
    "resnet50_classification_report.csv",
    "benchmarking_3_modeles.png",
    "benchmarking_3_modeles.csv",
    "benchmarking_temps_vs_accuracy.png",
]
for fname in files:
    status = "✅" if os.path.exists(fname) else "⏳"
    size   = f"({get_size_mb(fname)} MB)" if os.path.exists(fname) else ""
    print(f"  {status}  {fname}  {size}")
print("=" * 60)


## 🔬 PARTIE 9 — Analyse professionnelle

### 9.1 — Diagnostic automatique


In [ ]:
# ─── Diagnostic overfitting / généralisation ──────────────────────────────────
final_train_acc  = history_full["accuracy"][-1]
final_val_acc    = history_full["val_accuracy"][-1]
acc_gap          = final_train_acc - final_val_acc
final_train_loss = history_full["loss"][-1]
final_val_loss   = history_full["val_loss"][-1]

last_n = min(5, len(history_full["val_loss"]))
val_loss_trend = history_full["val_loss"][-last_n] - history_full["val_loss"][-1]

print("=" * 70)
print("DIAGNOSTIC AUTOMATIQUE — ResNet50 Fine-Tuned")
print("=" * 70)
print(f"  Accuracy finale Train       : {final_train_acc:.4f}  ({final_train_acc*100:.2f}%)")
print(f"  Accuracy finale Validation  : {final_val_acc:.4f}  ({final_val_acc*100:.2f}%)")
print(f"  Écart Train − Validation    : {acc_gap:.4f}")
print(f"  Loss finale Train           : {final_train_loss:.4f}")
print(f"  Loss finale Validation      : {final_val_loss:.4f}")
print()

if acc_gap > 0.10:
    print("⚠️  DIAGNOSTIC : Overfitting significatif (écart > 10%).")
    print("   ► Recommandations :")
    print("     - Augmenter Dropout (0.5 → 0.6)")
    print("     - Réduire le nombre de couches dégelées (30 → 20)")
    print("     - Ajouter kernel_regularizer=l2(1e-4) sur les Dense")
    print("     - Renforcer la Data Augmentation")
elif acc_gap > 0.05:
    print("🟡 DIAGNOSTIC : Overfitting léger (5–10%). Performances acceptables.")
    print("   ► Surveiller sur un run plus long ou tester Dropout(0.6).")
else:
    print("✅ DIAGNOSTIC : Bonne généralisation (écart < 5%).")
    print("   ResNet50 capture les patterns PlantVillage sans mémoriser le train set.")

print()
# Résumé comparatif final
print("─" * 70)
print("  RÉSUMÉ COMPARATIF FINAL")
print("─" * 70)
print(f"  CNN From Scratch       → Accuracy : {cnn_metrics.get('test_accuracy',0):.4f} | F1 : {cnn_metrics.get('f1_weighted',0):.4f}")
print(f"  MobileNetV2 Fine-Tuned → Accuracy : {mobilenet_metrics.get('test_accuracy',0):.4f} | F1 : {mobilenet_metrics.get('f1_weighted',0):.4f}")
print(f"  ResNet50 Fine-Tuned    → Accuracy : {resnet_metrics['test_accuracy']:.4f} | F1 : {resnet_metrics['f1_weighted']:.4f}")
print("─" * 70)

best_acc = max(
    cnn_metrics.get("test_accuracy",0),
    mobilenet_metrics.get("test_accuracy",0),
    resnet_metrics["test_accuracy"]
)
winner = ["CNN From Scratch","MobileNetV2","ResNet50"][[
    cnn_metrics.get("test_accuracy",0),
    mobilenet_metrics.get("test_accuracy",0),
    resnet_metrics["test_accuracy"]
].index(best_acc)]
print(f"  🏆 Meilleure Accuracy : {winner} ({best_acc:.4f})")


### 9.2 — Analyse scientifique et professionnelle

---

#### 🔵 Architecture ResNet50 — Points forts

**1. Connexions résiduelles (skip connections)**  
L'innovation fondamentale de ResNet est de contourner le problème de la disparition du gradient dans les réseaux profonds. En ajoutant directement l'entrée `x` à la sortie du bloc `F(x)`, le gradient peut se propager sans dégradation jusqu'aux premières couches. Cette propriété rend ResNet50 nettement plus stable à entraîner qu'un réseau de même profondeur sans connexions résiduelles.

**2. Blocs Bottleneck (1×1 → 3×3 → 1×1)**  
Chaque bloc Bottleneck réduit d'abord la dimension des features (1×1 conv), applique la convolution spatiale efficace (3×3 conv), puis restaure la dimension (1×1 conv). Cela réduit le coût computationnel tout en maintenant une capacité de représentation élevée.

**3. Capacité représentationnelle supérieure**  
Avec ~25,6 M de paramètres (vs ~3,4 M pour MobileNetV2), ResNet50 dispose d'une capacité discriminante plus élevée, potentiellement avantageuse pour distinguer 38 classes fines avec des symptômes visuellement proches (ex : différents stades de la même maladie).

---

#### 🟠 Comparaison ResNet50 vs MobileNetV2

| Dimension | MobileNetV2 | ResNet50 | Avantage |
|---|---|---|---|
| **Accuracy (attendue)** | ~90–95% | ~92–97% | ResNet50 (légèrement) |
| **Capacité (paramètres)** | 3,4 M | 25,6 M | ResNet50 (plus riche) |
| **Vitesse d'entraînement** | Rapide | Modérée | MobileNetV2 |
| **Déployabilité mobile** | ✅ Optimisé mobile | ⚠️ Lourd pour mobile | MobileNetV2 |
| **Robustesse aux classes fines** | Bonne | Meilleure | ResNet50 |
| **Risque d'overfitting** | Modéré | Légèrement plus élevé | MobileNetV2 |
| **Empreinte mémoire** | Faible (~14 MB) | Élevée (~100 MB) | MobileNetV2 |

**Recommandation de déploiement :**  
- Application mobile ou edge device → **MobileNetV2**  
- Serveur de diagnostic agricole (cloud / API) → **ResNet50**  
- Contrainte de précision maximale → **ResNet50**

---

#### 🟢 Impact du Fine-Tuning sur ResNet50

Le Fine-Tuning des 30 dernières couches (Stage 5 + fin du Stage 4) permet d'adapter les représentations haut niveau de ResNet50 au domaine botanique. On s'attend à un gain de +2 à +6% d'accuracy par rapport à la seule Feature Extraction (Phase 1), grâce à la spécialisation progressive des features profondes.

**Points de vigilance :**
- ResNet50 contient des couches BatchNorm dans la base. En mode `training=False` lors du passage dans la base (Feature Extraction), ces BN restent gelées (comportement recommandé). Lors du Fine-Tuning, certains frameworks les remettent en mode entraînement — surveiller que `base_model(inputs, training=False)` est bien maintenu.
- Le LR = 1e-5 est critique : trop élevé → catastrophic forgetting des poids Stage 5 ; trop bas → convergence très lente.

---

#### 📌 Recommandations pour la suite du projet

1. **Choix du modèle final** : Sélectionner le modèle avec le meilleur F1-Score weighted pour la suite (généralement ResNet50 ou MobileNetV2 selon les résultats obtenus).

2. **Étape 6 suggérée — Ensemble / Stacking** : Combiner les prédictions de MobileNetV2 et ResNet50 (average ou weighted voting) pour potentiellement gagner +1 à +2% d'accuracy supplémentaire.

3. **Analyse des erreurs résiduelles** : Les confusions persistantes identifiées dans les matrices des deux modèles (MobileNetV2 et ResNet50) méritent une analyse qualitative — visualisation des images mal classées, éventuelle collecte de données supplémentaires pour ces classes.

4. **Déploiement** : MobileNetV2 (`.tflite` via TensorFlow Lite) pour une application Android/iOS, ResNet50 (FastAPI + Docker) pour une API de diagnostic serveur.

---

#### 🏁 Conclusion scientifique

Ce benchmarking à trois voies (CNN From Scratch → MobileNetV2 → ResNet50) démontre empiriquement la hiérarchie classique en Deep Learning appliqué :

1. **CNN From Scratch** : baseline de référence — prouve que le problème est learnable, mais limité par la taille du dataset et l'absence de connaissances préalables.
2. **MobileNetV2 Fine-Tuned** : le meilleur rapport performance/coût — exploitation intelligente de 1,28 M d'images ImageNet pour compenser la taille modeste de PlantVillage.
3. **ResNet50 Fine-Tuned** : capacité de représentation maximale — meilleure performance absolue attendue, au prix d'un modèle plus lourd et d'un entraînement plus long.

Le Transfer Learning, dans les deux cas, réduit drastiquement le besoin de données labellisées et le temps de convergence, tout en améliorant la généralisation — conformément aux principes fondamentaux établis par (Yosinski et al., 2014) et (Pan & Yang, 2010).
